<a href="https://colab.research.google.com/github/anacvetkoo/MOS-projekt/blob/main/notebooks/ACS_priprava_IT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Priprava podatkov IPUMS ACS – računalniški poklici

## Namen

V tem zvezku sem pripravila podatkovno zbirko za analizo zaslužkov zaposlenih v računalniških poklicih glede na doseženo izobrazbo (RV5).

Uporabila sem podatke IPUMS USA – American Community Survey (ACS) za leta 2019, 2022 in 2024.

## Postopek

Iz celotnega razpoložljivega vzorca ACS sem izbrala posameznike, ki opravljajo računalniške poklice iz skupine SOC 15-12.

Zaradi velikosti izvorne datoteke sem podatke obdelala po delih in jih shranila v manjšo zbirko.

## Rezultat

Pripravila sem datoteko `ACS_2019_2022_2024_IT.csv`, ki vsebuje izbrane računalniške poklice za vsa tri leta.

Izvorno podatkovno zbirko sem ohranila nespremenjeno.

Pripravljena zbirka bo uporabljena pri analizi RV5 in po potrebi kot podatkovna podlaga za finančno simulacijo RV6.

In [1]:
# 1. Uvoz knjižnic in povezava z Google Drive

import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# Določitev poti do podatkov
mapa = Path(
    "/content/drive/MyDrive/FERI/3. letnik/1. semester/MOS/Projekt_teden2"
)

pot_acs = mapa / "ACS_2019_2022_2024_FULL.csv"

# Preverjanje obstoja datoteke
if not pot_acs.is_file():
    raise FileNotFoundError(f"Datoteka ni najdena: {pot_acs}")

# Preberem samo imena stolpcev, ne celotne velike datoteke
stolpci = pd.read_csv(pot_acs, nrows=0).columns.tolist()

# Izpis osnovnih informacij o datoteki
print("Datoteka:", pot_acs.name)
print("Velikost (GB):", round(pot_acs.stat().st_size / 1024**3, 2))
print("Število stolpcev:", len(stolpci))
print("Stolpci:", stolpci)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Datoteka: ACS_2019_2022_2024_FULL.csv
Velikost (GB): 1.0
Število stolpcev: 21
Stolpci: ['YEAR', 'SAMPLE', 'SERIAL', 'CBSERIAL', 'HHWT', 'CLUSTER', 'STRATA', 'GQ', 'PERNUM', 'PERWT', 'AGE', 'EDUC', 'EDUCD', 'DEGFIELD', 'DEGFIELDD', 'EMPSTAT', 'EMPSTATD', 'OCC', 'OCCSOC', 'UHRSWORK', 'INCWAGE']


In [2]:
# 2. Izbor računalniških poklicev in shranjevanje podatkov

# Izbor potrebnih spremenljivk
stolpci_it = [
    "YEAR", "SAMPLE", "AGE",
    "OCC", "OCCSOC",
    "EDUC", "EDUCD",
    "DEGFIELD", "DEGFIELDD",
    "EMPSTAT", "EMPSTATD",
    "UHRSWORK", "INCWAGE", "PERWT"
]

# Določitev izhodnih datotek
pot_acs_it = mapa / "ACS_2019_2022_2024_IT.csv"
# Začasna datoteka prepreči prepis končne datoteke med obdelavo
pot_acs_it_zacasna = mapa / "ACS_IT_zacasna.csv"

# Priprava na obdelavo
povzetki = []
prvi_del = True
pot_acs_it_zacasna.unlink(missing_ok=True)

# Branje in filtriranje podatkov po delih
for kos in pd.read_csv(
    pot_acs,
    usecols=stolpci_it,           # prebere samo spremenljivke ki jih potrebujem
    dtype={"OCCSOC": "string"},
    chunksize=100_000             # datoteko obdela po delih, namesto da bi jo v celoti naložila v pomnilnik
):

    # Poenotenje poklicnih kod
    # Odstranim presledke in vezaje, da imajo poklicne kode enoten zapis
    kode = kos["OCCSOC"].str.strip().str.replace("-", "", regex=False)

    # Izbor računalniških poklicev SOC 15-12
    maska = kode.str.startswith("1512", na=False)
    it = kos.loc[maska].copy()
    it["OCCSOC"] = kode.loc[maska]

    if it.empty:
        continue

    # Shranjevanje izbranih zapisov
    it.to_csv(
        pot_acs_it_zacasna,
        mode="w" if prvi_del else "a",
        header=prvi_del,
        index=False
    )

    prvi_del = False

    # Štetje zapisov po letih in poklicnih kodah
    povzetki.append(it.groupby(["YEAR", "OCCSOC"]).size())

# Preverjanje rezultatov
if prvi_del:
    raise ValueError("Ni bilo najdenih računalniških poklicev.")

pregled_kod = (
    pd.concat(povzetki)
    .groupby(level=["YEAR", "OCCSOC"])
    .sum()
    .reset_index(name="Stevilo")
)

if set(pregled_kod["YEAR"]) != {2019, 2022, 2024}:
    raise ValueError("V filtriranih podatkih niso prisotna vsa tri leta.")

# Shranjevanje končne datoteke
pot_acs_it_zacasna.replace(pot_acs_it)

# Izpis rezultatov
print("Število zapisov po letih:")
display(pregled_kod.groupby("YEAR")["Stevilo"].sum())

print("\nŠtevilo različnih poklicnih kod po letih:")
display(pregled_kod.groupby("YEAR")["OCCSOC"].nunique())

print("\nSkupno število zapisov:", pregled_kod["Stevilo"].sum())

print("Velikost nove datoteke (MB):",
      round(pot_acs_it.stat().st_size / 1024**2, 2))

print("Shranjeno v:", pot_acs_it)

Število zapisov po letih:


,Stevilo
YEAR,
2019,54127
2022,61069
2024,65474



Število različnih poklicnih kod po letih:


,OCCSOC
YEAR,
2019,13
2022,13
2024,13



Skupno število zapisov: 180670
Velikost nove datoteke (MB): 10.22
Shranjeno v: /content/drive/MyDrive/FERI/3. letnik/1. semester/MOS/Projekt_teden2/ACS_2019_2022_2024_IT.csv


### Rezultat priprave podatkov

Iz celotnega razpoložljivega vzorca IPUMS ACS za leta 2019, 2022 in 2024 sem izbrala posameznike z računalniškimi poklici iz skupine SOC 15-12.

Zaradi velikosti izvorne datoteke (približno 1 GB) sem podatke obdelala po delih po 100.000 vrstic. Pri filtriranju sem uporabila spremenljivko `OCCSOC` in ohranila samo spremenljivke, ki jih potrebujem za nadaljnjo analizo.

**Rezultati filtriranja:**

| Leto | Število zapisov |
|---|---:|
| 2019 | 54.127 |
| 2022 | 61.069 |
| 2024 | 65.474 |
| **Skupaj** | **180.670** |

V vseh treh letih sem identificirala 13 različnih poklicnih kod znotraj skupine SOC 15-12.

Pripravljene podatke sem shranila v datoteko `ACS_2019_2022_2024_IT.csv`, ki je velika približno 10,22 MB.

**Metodološka opomba:** V tej fazi sem podatke filtrirala samo glede na poklic. Posameznikov še nisem omejevala glede na starost, izobrazbo, zaposlitveni status ali zaslužke. Te omejitve sem uporabila pozneje pri analizi RV5 v zvezku `Ziva_teden2.ipynb`.

Izbrana poklicna skupina predstavlja računalniške poklice in ne celotnega IT-sektorja.